<a href="https://colab.research.google.com/github/WithAllMyHeart/Review-Uploader/blob/main/Review_Uploader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#!/usr/bin/env python3

import os
import requests
import sys

# Directory containing the feedback files
feedback_directory = "/data/feedback"

# API endpoint for feedback submission
feedback_api_url = "http://<corpweb-external-IP>/feedback"  # Replace with actual IP

def read_feedback_file(filepath):
    """Read a feedback file and return its content as a dictionary."""
    try:
        with open(filepath, "r") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]

        # Ensure the file has at least 4 lines (title, name, date, feedback)
        if len(lines) < 4:
            raise ValueError(f"File {filepath} does not contain enough lines (expected 4, got {len(lines)}).")

        feedback_dictionary = {
            'title': lines,
            'name': lines,
            'date': lines,
            'feedback': " ".join(lines[3:])
        }
        return feedback_dictionary

    except Exception as e:
        print(f"Error reading file {filepath}: {e}", file=sys.stderr)
        return None

def upload_feedback(data):
    """Upload feedback to the server and return the response."""
    try:
        response = requests.post(feedback_api_url, json=data)
        return response
    except requests.exceptions.RequestException as e:
        print(f"Error uploading feedback: {e}", file=sys.stderr)
        return None

def main():
    # List all files in the directory
    try:
        files = os.listdir(feedback_dir)
    except Exception as e:
        print(f"Error accessing directory {feedback_dir}: {e}", file=sys.stderr)
        sys.exit(1)

    # Iterate over each file
    for file in files:
        if file.endswith(".txt"):
            filepath = os.path.join(feedback_dir, file)
            print(f"Processing file: {file}")

            # Read the file and create the dictionary
            feedback_data = read_feedback_file(filepath)
            if not feedback_data:
                continue  # Skip if file reading failed

            # Upload the feedback
            response = upload_feedback(feedback_data)
            if not response:
                print("Failed to upload feedback due to server error.", file=sys.stderr)
                continue

            # Check if the upload was successful
            print(f"Uploaded {file}: Status Code {response.status_code}, Response: {response.text}")

            # Alert if the server is down (status code is not 201)
            if response.status_code != 201:
                print(f"⚠️ Alert: Server returned status code {response.status_code} for file {file}. Check the server.", file=sys.stderr)

if __name__ == "__main__":
    main()